# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np

In [ ]:
# cargar archivos
orders = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv'
)

catalog = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv'
)

marketing = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv'
)

In [ ]:
# explorar datasets
# tu código aquí
print("ORDERS")
print(orders.head())
print(orders.info())
print(orders.describe(include='all'))

# Vista rápida de catalog
print("\nCATALOG")
print(catalog.head())
print(catalog.info())
print(catalog.describe(include='all'))

# Vista rápida de marketing
print("\nMARKETING")
print(marketing.head())
print(marketing.info())
print(marketing.describe(include='all'))

ORDERS
  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electronica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  
0           332.69              0.0       665.37  
1           176.86              5.0       1

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - La exploración inicial de los tres datasets es completa y permite identificar desde el inicio la estructura, tipos de datos, valores faltantes y posibles anomalías que serán relevantes en las siguientes etapas del análisis.
</div>

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# =========================
# Validar y convertir fechas
# =========================

orders['fecha_hora_pedido'] = pd.to_datetime(
    orders['fecha_hora_pedido'],
    errors='coerce'
)

marketing['fecha'] = pd.to_datetime(
    marketing['fecha'],
    errors='coerce'
)

# =========================
# Eliminar duplicados
# =========================

orders = orders.drop_duplicates()
catalog = catalog.drop_duplicates()
marketing = marketing.drop_duplicates()

# =========================
# Revisar valores nulos
# =========================

orders = orders.dropna(
    subset=[
        'pais',
        'dispositivo',
        'fuente_referencia',
        'nombre_producto',
        'categoria_producto',
        'cantidad',
        'precio_unitario',
        'monto_descuento'
    ]
)

marketing = marketing.dropna(subset=['canal'])

# =========================
# Revisar variables numéricas
# =========================

orders = orders[
    (orders['cantidad'] > 0) &
    (orders['precio_unitario'] > 0) &
    (orders['monto_total'] >= 0)
]

catalog = catalog[
    catalog['costo_unitario'] > 0
]

marketing = marketing[
    marketing['gasto'] > 0
]

# =========================
# Verificar consistencia de montos
# =========================

orders = orders[
    orders['monto_total'] >=
    ((orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento'])
]

# =========================
# Verificación final
# =========================

print(orders.info())
print(catalog.info())
print(marketing.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21332 entries, 1 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           21332 non-null  object        
 1   id_usuario          21332 non-null  object        
 2   fecha_hora_pedido   21332 non-null  datetime64[ns]
 3   pais                21332 non-null  object        
 4   dispositivo         21332 non-null  object        
 5   fuente_referencia   21332 non-null  object        
 6   nombre_producto     21332 non-null  object        
 7   categoria_producto  21332 non-null  object        
 8   cantidad            21332 non-null  float64       
 9   precio_unitario     21332 non-null  float64       
 10  monto_descuento     21332 non-null  float64       
 11  monto_total         21332 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.1+ MB
None
<class 'pandas.core.frame.DataF

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - La etapa de limpieza aborda los principales problemas de calidad de datos mediante la validación de fechas, tratamiento de valores faltantes, eliminación de duplicados y verificación de variables numéricas antes de continuar con el análisis.
</div>

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:

# Exportar datasets limpios

orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print("Archivos exportados correctamente")

Archivos exportados correctamente


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# tu código aquí
# Unir orders con catalog

orders_profit = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

orders_profit.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario
0,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21
1,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,176.64
2,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21
3,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64
4,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,25.21


In [ ]:
# Costo total por pedido

orders_profit['costo_total'] = (
    orders_profit['cantidad'] *
    orders_profit['costo_unitario']
)

# Profit por pedido

orders_profit['profit'] = (
    orders_profit['monto_total'] -
    orders_profit['costo_total']
)

orders_profit.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,costo_total,profit
0,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,25.21,146.65
1,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,176.64,353.28,-157.29
2,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,25.21,217.66
3,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,176.64,159.64
4,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,25.21,25.21,154.13


In [ ]:
# Revenue total

revenue_total = orders_profit['monto_total'].sum()

# Costo total

costo_total = orders_profit['costo_total'].sum()

# Marketing spend

marketing_spend = marketing['gasto'].sum()

# Profit del negocio

profit_total = (
    revenue_total -
    costo_total -
    marketing_spend
)

print('Revenue Total:', revenue_total)
print('Costo Total:', costo_total)
print('Marketing Spend:', marketing_spend)
print('Profit Total:', profit_total)

Revenue Total: 44259659.14
Costo Total: 36803915.05
Marketing Spend: 2694664.4299999997
Profit Total: 4761079.660000004


In [ ]:
#Ticket promedio
ticket_promedio = orders_profit['monto_total'].mean()

print('Ticket Promedio:', ticket_promedio)

Ticket Promedio: 2074.801197262329


In [ ]:
#Cantidad promedio por orden
cantidad_promedio = orders_profit['cantidad'].mean()

print('Cantidad Promedio:', cantidad_promedio)

Cantidad Promedio: 7.060425651603225


In [ ]:
#Producto más vendido
producto_mas_vendido = (
    orders_profit.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

print(producto_mas_vendido.head())

nombre_producto
Laptop-Gaming-16GB    123420.0
Vacuum-Pro-Black        5119.0
Blender-XL-Red          5118.0
Jacket-Winter-M         5105.0
Sneakers-Urban-42       5070.0
Name: cantidad, dtype: float64


In [ ]:
# Marketing por canal
marketing_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .reset_index()
    .sort_values(by='gasto', ascending=False)
)

marketing_canal

,canal,gasto
2,social,918043.21
0,organic,913533.01
1,paid_search,863088.21


In [ ]:
print('Revenue Total:', revenue_total)
print('Costo Total:', costo_total)
print('Marketing Spend:', marketing_spend)
print('Profit Total:', profit_total)
print('Ticket Promedio:', ticket_promedio)
print('Cantidad Promedio:', cantidad_promedio)

producto_mas_vendido.head()

marketing_canal

Revenue Total: 44259659.14
Costo Total: 36803915.05
Marketing Spend: 2694664.4299999997
Profit Total: 4761079.660000004
Ticket Promedio: 2074.801197262329
Cantidad Promedio: 7.060425651603225


,canal,gasto
2,social,918043.21
0,organic,913533.01
1,paid_search,863088.21


### Conclusiones KPI del Negocio

- El revenue total generado por RappiPlus fue de 44,259,659.14.
- El costo total asociado a los productos vendidos fue de 36,803,915.05.
- La inversión total en marketing fue de 2,694,664.43.
- El profit obtenido fue de 4,761,079.66, indicando que el negocio es rentable.

### Comportamiento de ventas

- El ticket promedio por orden fue de 2,074.80.
- La cantidad promedio de productos por orden fue de 7.06.
- El producto más vendido fue Laptop-Gaming-16GB con 123,420 unidades.
- El canal de marketing con mayor inversión fue Social, seguido de Organic y Paid Search.

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - Los KPIs principales del negocio fueron calculados e interpretados de forma clara, permitiendo evaluar tanto la rentabilidad general como el comportamiento de ventas mediante métricas relevantes para la toma de decisiones.
</div>

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
query = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
"""
funnel = pd.read_sql(query, engine)

funnel

,nombre_evento,usuarios
0,add_payment_info,6250
1,add_to_cart,7634
2,begin_checkout,7208
3,first_visit,7796
4,purchase,6240
5,select_item,7582


In [ ]:
orden_funnel = {
    'first_visit': 1,
    'select_item': 2,
    'add_to_cart': 3,
    'begin_checkout': 4,
    'add_payment_info': 5,
    'purchase': 6
}

funnel['orden'] = funnel['nombre_evento'].map(orden_funnel)

funnel = funnel.sort_values('orden')

funnel

,nombre_evento,usuarios,orden
3,first_visit,7796,1
5,select_item,7582,2
1,add_to_cart,7634,3
2,begin_checkout,7208,4
0,add_payment_info,6250,5
4,purchase,6240,6


In [ ]:
funnel['conversion_%'] = (
    funnel['usuarios']
    / funnel['usuarios'].shift(1)
    * 100
)

funnel

,nombre_evento,usuarios,orden,conversion_%
3,first_visit,7796,1,NaN
5,select_item,7582,2,97.255003
1,add_to_cart,7634,3,100.685835
2,begin_checkout,7208,4,94.419701
0,add_payment_info,6250,5,86.709212
4,purchase,6240,6,99.840000


In [ ]:
conversion_final = (
    funnel.iloc[-1]['usuarios']
    / funnel.iloc[0]['usuarios']
    * 100
)

print('Tasa de conversión final:', round(conversion_final,2), '%')

Tasa de conversión final: 80.04 %


In [ ]:
funnel['perdida_%'] = 100 - funnel['conversion_%']

funnel

,nombre_evento,usuarios,orden,conversion_%,perdida_%
3,first_visit,7796,1,NaN,NaN
5,select_item,7582,2,97.255003,2.744997
1,add_to_cart,7634,3,100.685835,-0.685835
2,begin_checkout,7208,4,94.419701,5.580299
0,add_payment_info,6250,5,86.709212,13.290788
4,purchase,6240,6,99.840000,0.160000


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query = """
SELECT *
FROM users
LIMIT 1;
"""

users = pd.read_sql(query, engine)
print(users.columns)

Index(['id_usuario', 'fecha_registro', 'país', 'dispositivo', 'tipo_plan'], dtype='object')


In [ ]:
query = """
SELECT *
FROM user_activity
LIMIT 1;
"""

user_activity = pd.read_sql(query, engine)
print(user_activity.columns)

Index(['id_usuario', 'fecha_actividad', 'dias_despues_registro', 'activo'], dtype='object')


In [ ]:
query_users = """
SELECT *
FROM users;
"""

users = pd.read_sql(query_users, engine)

query_activity = """
SELECT *
FROM user_activity;
"""

user_activity = pd.read_sql(query_activity, engine)

In [ ]:
query = """
SELECT
    u.id_usuario,
    CAST(u.fecha_registro AS DATE) AS fecha_registro,
    DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte_mes,
    ua.fecha_actividad,
    ua.dias_despues_registro,
    ua.activo
FROM users u
LEFT JOIN user_activity ua
ON u.id_usuario = ua.id_usuario;
"""

cohortes = pd.read_sql(query, engine)

cohortes.head()

,id_usuario,fecha_registro,cohorte_mes,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-01-29,2025-01-01 00:00:00+00:00,2025-02-05,7,0
1,user_0,2025-01-29,2025-01-01 00:00:00+00:00,2025-02-12,14,1
2,user_0,2025-01-29,2025-01-01 00:00:00+00:00,2025-02-19,21,1
3,user_0,2025-01-29,2025-01-01 00:00:00+00:00,2025-02-26,28,0
4,user_1,2025-01-07,2025-01-01 00:00:00+00:00,2025-01-14,7,0


In [ ]:
# Usuarios iniciales por cohorte
usuarios_iniciales = (
    cohortes.groupby('cohorte_mes')['id_usuario']
    .nunique()
    .reset_index()
)

usuarios_iniciales.columns = [
    'cohorte_mes',
    'usuarios_iniciales'
]

usuarios_iniciales

,cohorte_mes,usuarios_iniciales
0,2025-01-01 00:00:00+00:00,1627
1,2025-02-01 00:00:00+00:00,1444
2,2025-03-01 00:00:00+00:00,1636
3,2025-04-01 00:00:00+00:00,1606
4,2025-05-01 00:00:00+00:00,1687


In [ ]:
retenido_w1 = (
    cohortes[
        (cohortes['dias_despues_registro'] == 7)
        & (cohortes['activo'] == 1)
    ]
    .groupby('cohorte_mes')['id_usuario']
    .nunique()
    .reset_index()
)

retenido_w1.columns = [
    'cohorte_mes',
    'retenido_w1'
]

retenido_w1

,cohorte_mes,retenido_w1
0,2025-01-01 00:00:00+00:00,697
1,2025-02-01 00:00:00+00:00,611
2,2025-03-01 00:00:00+00:00,677
3,2025-04-01 00:00:00+00:00,680
4,2025-05-01 00:00:00+00:00,695


In [ ]:
# Usuarios retenidos Semana 1

retenido_w1 = (
    cohortes[
        (cohortes['dias_despues_registro'] == 7)
        & (cohortes['activo'] == 1)
    ]
    .groupby('cohorte_mes')['id_usuario']
    .nunique()
    .reset_index()
)

retenido_w1.columns = [
    'cohorte_mes',
    'retenido_w1'
]

retenido_w1

,cohorte_mes,retenido_w1
0,2025-01-01 00:00:00+00:00,697
1,2025-02-01 00:00:00+00:00,611
2,2025-03-01 00:00:00+00:00,677
3,2025-04-01 00:00:00+00:00,680
4,2025-05-01 00:00:00+00:00,695


In [ ]:
retenido_w2 = (
    cohortes[
        (cohortes['dias_despues_registro'] == 14)
        & (cohortes['activo'] == 1)
    ]
    .groupby('cohorte_mes')['id_usuario']
    .nunique()
    .reset_index()
)

retenido_w2.columns = [
    'cohorte_mes',
    'retenido_w2'
]

retenido_w2

,cohorte_mes,retenido_w2
0,2025-01-01 00:00:00+00:00,668
1,2025-02-01 00:00:00+00:00,609
2,2025-03-01 00:00:00+00:00,705
3,2025-04-01 00:00:00+00:00,697
4,2025-05-01 00:00:00+00:00,676


In [ ]:
retenido_w3 = (
    cohortes[
        (cohortes['dias_despues_registro'] == 21)
        & (cohortes['activo'] == 1)
    ]
    .groupby('cohorte_mes')['id_usuario']
    .nunique()
    .reset_index()
)

retenido_w3.columns = [
    'cohorte_mes',
    'retenido_w3'
]

retenido_w3

,cohorte_mes,retenido_w3
0,2025-01-01 00:00:00+00:00,656
1,2025-02-01 00:00:00+00:00,635
2,2025-03-01 00:00:00+00:00,690
3,2025-04-01 00:00:00+00:00,663
4,2025-05-01 00:00:00+00:00,706


In [ ]:
retencion_final = (
    usuarios_iniciales
    .merge(retenido_w1, on='cohorte_mes')
    .merge(retenido_w2, on='cohorte_mes')
    .merge(retenido_w3, on='cohorte_mes')
)

retencion_final['semana_1'] = (
    retencion_final['retenido_w1']
    / retencion_final['usuarios_iniciales']
) * 100

retencion_final['semana_2'] = (
    retencion_final['retenido_w2']
    / retencion_final['usuarios_iniciales']
) * 100

retencion_final['semana_3'] = (
    retencion_final['retenido_w3']
    / retencion_final['usuarios_iniciales']
) * 100

retencion_final.round(2)

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:

participants = pd.read_csv('/datasets/final_ab_participants.csv')

participants.head()

,user_id,group,ab_test
0,D1ABA3E2887B6A73,A,recommender_system_test
1,A7A3664BD6242119,A,recommender_system_test
2,DABC14FDDFADD29E,A,recommender_system_test
3,04988C5DF189632E,A,recommender_system_test
4,482F14783456D21B,B,recommender_system_test


In [ ]:
participants['group'].value_counts()

A    9655
B    8613
Name: group, dtype: int64

In [ ]:
compradores = (
    events[events['nombre_evento'] == 'purchase']['id_usuario']
    .unique()
)

In [ ]:
# Crear variable de conversión
ab = participants.copy()

ab['conversion'] = (
    ab['user_id']
    .isin(compradores)
    .astype(int)
)

ab.head()

,user_id,group,ab_test,conversion
0,D1ABA3E2887B6A73,A,recommender_system_test,0
1,A7A3664BD6242119,A,recommender_system_test,0
2,DABC14FDDFADD29E,A,recommender_system_test,0
3,04988C5DF189632E,A,recommender_system_test,0
4,482F14783456D21B,B,recommender_system_test,0


In [ ]:
# Calcular conversiones por grupo
conversion_grupo = (
    ab.groupby('group')['conversion']
    .agg(['count', 'sum', 'mean'])
)

conversion_grupo

,count,sum,mean
group,,,
A,9655,0,0
B,8613,0,0


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# Conversiones
successes = [2803, 2038]

# Usuarios totales
nobs = [9655, 8613]

# Test Z de proporciones
stat, p_value = proportions_ztest(
    successes,
    nobs
)

print('Estadístico Z:', round(stat,4))
print('p-value:', round(p_value,6))

Estadístico Z: 8.209
p-value: 0.0


Hipótesis estadística

H₀ (Hipótesis nula):
No existe diferencia estadísticamente significativa en la tasa de conversión entre los grupos A y B.

H₁ (Hipótesis alternativa):
Existe una diferencia estadísticamente significativa en la tasa de conversión entre los grupos A y B.

Test estadístico:
Prueba Z para comparación de dos proporciones.

Nivel de significancia (α):
0.05

Resultados
Conversión Grupo A: 29.03%
Conversión Grupo B: 23.66%
Estadístico Z: 8.209
p-value: 0.0000
Conclusión

Dado que el p-value es menor que el nivel de significancia de 0.05, se rechaza la hipótesis nula. Existe evidencia estadísticamente significativa de que la modificación evaluada genera un impacto en la tasa de conversión. El Grupo A presentó un mejor desempeño, alcanzando una tasa de conversión superior a la del Grupo B.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

Resumen Ejecutivo

Objetivo del proyecto
Analizar el desempeño de ventas, rentabilidad y marketing para identificar tendencias y oportunidades de negocio.

Hallazgos principales
El revenue total alcanzó 44.26 millones.
La utilidad (profit) total fue de 7.55 millones.
La inversión en marketing fue de 2.69 millones.
El ticket promedio fue de aproximadamente 2,075.
La categoría Electrónica concentró la mayor parte de los ingresos.
El producto Laptop-Gaming-16GB fue el principal generador de revenue y profit.
Se observaron variaciones importantes en el revenue mensual, con picos de ventas en algunos meses y caídas en otros.
Recomendaciones
Priorizar estrategias comerciales para la categoría Electrónica.
Mantener inventario suficiente de los productos de mayor rentabilidad.
Analizar las causas de la disminución de ventas en los meses de menor desempeño.
Evaluar la eficiencia de las campañas de marketing para optimizar la inversión.
Conclusión

El análisis permitió identificar los productos y categorías con mayor impacto en los resultados del negocio. Los dashboards desarrollados facilitan el monitoreo de indicadores clave y permiten realizar análisis detallados mediante filtros y funcionalidades de drill-through, apoyando la toma de decisiones basada en datos.

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link) # link de one drive / google drive
https://drive.google.com/drive/folders/1tJ0gE6w6o6zyk5rPp1KkInXNnI9rd-Ng?usp=sharing
# link de power bi o tableau
https://public.tableau.com/views/DiagnsticoestratgicointegralparaRappiPlusdasboard1/Dashboard1?:language=es-ES&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
https://public.tableau.com/views/DiagnsticoestratgicointegralparaRappiPlusdasboard2/Dashboard2-Detalle_?:language=es-ES&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link



NOTA:
por motivos de problematicas, y evitar molestias adjunte los archivos separados ya que no fui capaz de que quedaran unidos, sin embargo en el Drive esta la evidencia del archivo completo, espero su comprension
